

## Semantic-Based Topic Modeling

In [1]:
from datetime import datetime
date = datetime.now()
formatted_date = date.strftime("%B %d, %Y")
print(formatted_date)

July 13, 2024


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
userdata.get('HF_TOKEN')

# Set up the current working directory within the Google Drive
%cd /content/drive/My\ Drive/Colab\ Notebooks/LLM/sped_biblio/topic_modeling

Mounted at /content/drive
/content/drive/My Drive/Colab Notebooks/LLM/sped_biblio/topic_modeling


In [3]:
# !pip install -q pandas numpy sentence-transformers bertopic scikit-learn matplotlib umap-learn hdbscan
!pip install -q qgrid nltk sentence_transformers bertopic umap-learn hdbscan dill networkx tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.2/889.2 kB 17.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 29.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 kB 22.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 19.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 91.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 17.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.5 MB/s eta 0:00:00


In [4]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='nltk')

import re
import warnings
from collections import defaultdict
import pickle
from pickle import UnpicklingError

# Data Manipulation
import dill
import numpy as np
import pandas as pd
import requests
import qgrid

# Natural Language Processing
import nltk
from nltk.stem import WordNetLemmatizer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertModel, BertTokenizer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

# Clustering
from hdbscan import HDBSCAN
from umap import UMAP
from scipy.cluster import hierarchy as sch

# Visualization Imports
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.ticker import FuncFormatter
import colorlover as cl
import plotly.io as pio

# Network Analysis
import networkx as nx

# Progress Bar
from tqdm import tqdm

# Display HTML
from IPython.display import IFrame

nltk.download('wordnet')
pio.renderers.default = "colab"

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
[nltk_data] Downloading package wordnet to /root/nltk_data...


#### Combine text columns

In [5]:
all_data_file = f"files/all_data.xlsx"
all_data = pd.read_excel(all_data_file, na_filter=False)

df = all_data[all_data['filtered'] == 'Yes']
df['Year'] = df['PY'].astype(int)
df['Decade'] = (df['Year'] // 10) * 10
df['CR'] = df['CR'].astype(str)

#### Cluster documents

In [6]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
sentence_embeddings = sentence_model.encode(df['combined_text'].tolist(), show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/102 [00:00<?, ?it/s]

In [7]:
umap_model = UMAP(n_neighbors=5, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
reduced_embeddings = umap_model.fit_transform(sentence_embeddings)
hdbscan_model = HDBSCAN(min_cluster_size=35, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
hdbscan_model.fit(reduced_embeddings)
labels = hdbscan_model.labels_

In [ ]:
# Map embeddings into a 3D space
df_cluster = pd.DataFrame(np.hstack([reduced_embeddings, labels.reshape(-1, 1)]),
     columns=["x", "y", "cluster"]).sort_values("cluster")

# Visualize clusters
df_cluster['cluster'] = df_cluster['cluster'].astype(int).astype(str)

# Create interactive plot with Plotly
fig_cluster = px.scatter(df_cluster, x='x', y='y', color='cluster',
                 title='',
                 labels={'x': 'X', 'y': 'Y'},
                 hover_name='cluster',
                 opacity=0.4)

fig_cluster.update_traces(marker=dict(size=5), selector=dict(mode='markers'))

fig_cluster.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=800,
    height=600,
    title={
        'y':0.9,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'}
)

fig_cluster.write_html("results/fig_cluster.html")
fig_cluster.show()

In [59]:
IFrame(src='./results/fig_cluster.html', width=1450, height=600)

#### Topic modeling

In [60]:
# Initialize and fit BERTopic
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(df['combined_text'])

In [ ]:
topic_model_fig = topic_model.visualize_topics()
topic_model_fig.write_html("results/topic_model_fig.html")
topic_model_fig.show()

In [62]:
IFrame(src='./results/topic_model_fig.html', width=1450, height=600)

In [63]:
lemmatizer = WordNetLemmatizer()
def lemmatize_text(text):
    tokens = text.split()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(lemmatized_tokens)

def preprocess_texts(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s-]', '', text)
    text = re.sub(r'\d+', '', text)
    tokens = text.split()
    return ' '.join(tokens)

df['lemmatized_text'] = df['combined_text'].apply(lemmatize_text)

df['preprocessed_text'] = df['lemmatized_text'].apply(preprocess_texts)

vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words='english')
ctfidf_model = ClassTfidfTransformer()
representation_model = KeyBERTInspired()

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  ctfidf_model=ctfidf_model,
  representation_model=representation_model,
  calculate_probabilities=True,
  verbose=True
)

topics, probs = topic_model.fit_transform(df['preprocessed_text'])

2024-07-13 04:50:10,759 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/102 [00:00<?, ?it/s]

2024-07-13 04:50:13,458 - BERTopic - Embedding - Completed ✓
2024-07-13 04:50:13,459 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-07-13 04:50:27,645 - BERTopic - Dimensionality - Completed ✓
2024-07-13 04:50:27,646 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-07-13 04:50:27,863 - BERTopic - Cluster - Completed ✓
2024-07-13 04:50:27,868 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-07-13 04:50:33,790 - BERTopic - Representation - Completed ✓


In [66]:
with open("files/topic_model.pkl", "wb") as f:
    pickle.dump(topic_model, f)

In [67]:
vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words='english')
topic_model.update_topics(df['preprocessed_text'], vectorizer_model=vectorizer_model)

In [68]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,900,-1_study_students_intervention_design,"[study, students, intervention, design, childr...",[A SYSTEMATIC REVIEW OF SINGLE-CASE RESEARCH S...
1,0,460,0_video_skills_social_autism,"[video, skills, social, autism, modeling, vide...",[A TWO EXPERIMENT TREATMENT COMPARISON STUDY T...
2,1,279,1_telehealth_behavior_parent_training,"[telehealth, behavior, parent, training, paren...",[TELEHEALTH AND AUTISM TREATING CHALLENGING BE...
3,2,262,2_reinforcement_behavior_stereotypy_response,"[reinforcement, behavior, stereotypy, response...",[A COMPARISON OF NONCONTINGENT REINFORCEMENT A...
4,3,192,3_communication_speech_autism_children,"[communication, speech, autism, children, aac,...",[A FURTHER COMPARISON OF MANUAL SIGNING PICTUR...
5,4,191,4_memory_design_study_intervention,"[memory, design, study, intervention, pain, si...",[EVALUATION OF THE EFFECTIVENESS OF A NOVEL BR...
6,5,162,5_reading_students_words_instruction,"[reading, students, words, instruction, vocabu...",[IPAD-ASSISTEDREADING FLUENCY INSTRUCTION FOR ...
7,6,153,6_training_teachers_feedback_instruction,"[training, teachers, feedback, instruction, st...",[TEACHING STATISTICAL VARIABILITY WITH EQUIVAL...
8,7,143,7_health_treatment_intervention_study,"[health, treatment, intervention, study, anxie...",[COGNITIVE-BEHAVIORAL THERAPY FOR ANXIETY IN P...
9,8,111,8_students_mathematics_manipulatives_math,"[students, mathematics, manipulatives, math, v...",[ADDING IT UP COMPARING CONCRETE AND APP-BASED...


In [69]:
# custom_labels = {topic: f"Topic {topic}" for topic in df['Topic'].unique()}

custom_labels = {
    -1: "topic -1",
    0: "topic 0",
    1: "topic 1",
    2: "topic 2",
    3: "topic 3",
    4: "topic 4",
    5: "topic 5",
    6: "topic 6",
    7: "topic 7",
    8: "topic 8",
    9: "topic 9",
    10: "topic 10",
    11: "topic 11",
    12: "topic 12",
    13: "topic 13",
    14: "topic 14"
}
topic_model.set_topic_labels(custom_labels)

In [ ]:
topic_word_barchart = topic_model.visualize_barchart(top_n_topics=15, n_words=10, custom_labels=True)

traces = topic_word_barchart.data

num_charts = len(traces)
num_columns = 3
num_rows = (num_charts + num_columns - 1) // num_columns

fig = make_subplots(
    rows=num_rows,
    cols=num_columns,
    vertical_spacing=0.07,
    subplot_titles=[custom_labels[i] for i in range(num_charts)]
)

def get_color_map(num_colors):
    base_colors = px.colors.qualitative.Set3
    if num_colors <= len(base_colors):
        return base_colors[:num_colors]
    else:
        extended_colors = base_colors * (num_colors // len(base_colors)) + base_colors[:num_colors % len(base_colors)]
        return extended_colors

# color_map = px.colors.qualitative.Pastel[:20]

color_map = [
    "#ABDEE6", "#FF968A", "#F2B2AC",
    "#F2C879","#FFA6C3", "#F7D9C4",
    "#D9C4B8", "#FFC8A2", "#D4F0F0",
    "#FAEDCB", "#DEFFC4", "#C6DEF1",
    "#A3C4F3", "#90DBF4", "#8EECF5",
]
df['Topic'] = topic_model.get_topic_info()['Topic']
# topics = df['Topic'].unique()
color_map_extended = color_map + color_map[:max(0, len(custom_labels) - len(color_map))]
topic_color_map = {i: color for i, color in enumerate(color_map_extended[:len(custom_labels)])}

for i, trace in enumerate(traces):
    row = (i // num_columns) + 1
    col = (i % num_columns) + 1
    topic_idx = i

    if topic_idx == -1:
        trace.marker.color = 'grey'
    else:
        trace.marker.color = topic_color_map[topic_idx]

    trace.update(width=0.8)
    trace.showlegend = False
    fig.add_trace(trace, row=row, col=col)

fig.update_layout(
    title_text="Topic Word Scores",
    margin=dict(t=80, b=80, l=80, r=80),
    width=1600,
    height=1200,
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)

fig.write_html("results/topic_word_barchart.html")
fig.show()

In [76]:
IFrame(src='./results/topic_word_barchart.html', width=1600, height=1200)

In [ ]:
heatmap = topic_model.visualize_heatmap(custom_labels=True, width=800, height=800)
heatmap.write_html("results/heatmap.html")
heatmap.show()

In [78]:
IFrame(src='./results/heatmap.html', width=800, height=800)

In [ ]:
# Hierarchical topics
linkage_function = lambda x: sch.linkage(x, 'single', optimal_ordering=True)
hierarchical_topics = topic_model.hierarchical_topics(df['preprocessed_text'], linkage_function=linkage_function)
hierarchical_topics_fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics, custom_labels=True, width=1000, height=1000)
hierarchical_topics_fig.write_html("results/hierarchical_topics_fig.html")
hierarchical_topics_fig.show()

In [80]:
IFrame(src='./results/hierarchical_topics_fig.html', width=1000, height=600)

In [81]:
def extract_ngrams(X, features):
    unigrams = []
    bigrams = []
    trigrams = []

    for row in X:
        present_ngrams = features[row.indices]
        unigrams.append([token for token in present_ngrams if len(token.split()) == 1])
        bigrams.append([token for token in present_ngrams if len(token.split()) == 2])
        trigrams.append([token for token in present_ngrams if len(token.split()) == 3])

    return unigrams, bigrams, trigrams

In [82]:
doc_info = topic_model.get_document_info(df['preprocessed_text'])
doc_info_df = pd.DataFrame(doc_info)
df = pd.concat([df.reset_index(drop=True), doc_info_df], axis=1)

vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words='english')
ngram_matrix = vectorizer_model.fit_transform(df['preprocessed_text'])
features = np.array(vectorizer_model.get_feature_names_out())
df['unigrams'], df['bigrams'], df['trigrams'] = extract_ngrams(ngram_matrix, features)

In [83]:
def extract_ngrams_per_topic(df, ngram_type='unigrams'):
    ngrams_per_topic = defaultdict(lambda: defaultdict(set))
    for custom_label in df['CustomName'].unique():
        topic_df = df[df['CustomName'] == custom_label]
        for Year in sorted(topic_df['Year'].unique()):
            year_df = topic_df[topic_df['Year'] == Year]
            for ngrams in year_df[ngram_type]:
                ngrams_per_topic[custom_label][Year].update(ngrams)
    return ngrams_per_topic

def calculate_proportion_new_ngrams_per_topic(ngrams_per_topic):
    proportions = []
    for CustomName, years_ngrams in ngrams_per_topic.items():
        previous_ngrams = set()
        for Year, ngrams in sorted(years_ngrams.items()):
            new_unique_ngrams = ngrams - previous_ngrams
            proportion_new_ngrams = len(new_unique_ngrams) / len(ngrams) if ngrams else 0
            previous_ngrams = previous_ngrams.union(ngrams)
            proportions.append({
                'CustomName': CustomName,
                'Year': Year,
                'unique_ngram_count': len(ngrams),
                'proportion_new_ngrams': proportion_new_ngrams
            })
    return pd.DataFrame(proportions)

def comma_formatter(value, _):
    return f"{value:,}"

def decimal_formatter(value, _):
    return f"{value:.2f}"

In [96]:
if 'CustomName' in df.columns:
  df = df.loc[:, ~df.columns.duplicated()]

df['Topic'] = topic_model.get_topic_info()['Topic'].astype(int)
df['CustomName'] = df['CustomName'].str.strip()
df_filtered = df[(df['CustomName'] != 'topic -1')]

In [98]:
unigrams_per_topic = extract_ngrams_per_topic(df_filtered, 'unigrams')
bigrams_per_topic = extract_ngrams_per_topic(df_filtered, 'bigrams')
trigrams_per_topic = extract_ngrams_per_topic(df_filtered, 'trigrams')

unigram_proportions_topic = calculate_proportion_new_ngrams_per_topic(unigrams_per_topic)
bigram_proportions_topic = calculate_proportion_new_ngrams_per_topic(bigrams_per_topic)
trigram_proportions_topic = calculate_proportion_new_ngrams_per_topic(trigrams_per_topic)

unigram_proportions_topic['ngram_type'] = 'unigram'
bigram_proportions_topic['ngram_type'] = 'bigram'
trigram_proportions_topic['ngram_type'] = 'trigram'

all_proportions_topic = pd.concat([unigram_proportions_topic, bigram_proportions_topic, trigram_proportions_topic])

In [ ]:
all_proportions_topic['Year'] = all_proportions_topic['Year'].astype(str)

colors = {
    'unigram': '#ff9d00',
    'bigram': '#1F78B4',
    'trigram': '#6A3D9A',
}

unique_topics = df_filtered.sort_values('Topic')['CustomName'].unique()
unique_topics = sorted(unique_topics, key=lambda x: int(x.split()[-1]))

num_cols = 3
num_rows = (len(unique_topics) // num_cols) + (1 if len(unique_topics) % num_cols != 0 else 0)

min_y = all_proportions_topic['unique_ngram_count'].min()
max_y = all_proportions_topic['unique_ngram_count'].max()

ngram_count_fig = make_subplots(
    rows=num_rows, cols=num_cols, subplot_titles=unique_topics,
    vertical_spacing=0.1, horizontal_spacing=0.1
)

legend_added = {'unigram': False, 'bigram': False, 'trigram': False}
for i, custom_label in enumerate(unique_topics):
    topic_df = all_proportions_topic[all_proportions_topic['CustomName'] == custom_label]
    row = (i // num_cols) + 1
    col = (i % num_cols) + 1
    for ngram_type in ['unigram', 'bigram', 'trigram']:
        filtered_df = topic_df[topic_df['ngram_type'] == ngram_type]
        ngram_count_fig.add_trace(
            go.Scatter(
                x=filtered_df['Year'],
                y=filtered_df['unique_ngram_count'],
                mode='lines+markers',
                name=ngram_type.capitalize() if not legend_added[ngram_type] else None,
                line=dict(color=colors[ngram_type], width=0.9),
                marker=dict(size=3),
                hovertemplate=(
                    'N-gram Type: %{text}<br>'
                    'Year: %{x}<br>'
                    'Custom Label: %{customdata}<br>'
                    'Count: %{y}<br>'
                ),
                text=filtered_df['ngram_type'],
                customdata=[custom_label] * len(filtered_df),
                showlegend=not legend_added[ngram_type]
            ),
            row=row,
            col=col
        )
        legend_added[ngram_type] = True

ngram_count_fig.update_layout(
    width=num_cols * 400,
    height=num_rows * 250,
    showlegend=True,
    legend_title_text='N-gram Type',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.1,
        xanchor="center",
        x=0.5
    ),
    title_text="Unique N-gram Counts Over Years by Topic",
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)

ngram_count_fig.add_annotation(
    x=-0.08, y=0.3,
    xref='paper', yref='paper',
    showarrow=False,
    textangle=-90,
    text="Unique N-grams"
)

ngram_count_fig.update_xaxes(tickangle=-70, showline=True, linewidth=0.7, linecolor='black', showticklabels=True)
ngram_count_fig.update_yaxes(showline=True, linewidth=0.7, linecolor='black', showticklabels=True, title_text=None)

ngram_count_fig.write_html("results/ngram_count_fig.html")
ngram_count_fig.show()

In [104]:
IFrame(src='./results/ngram_count_fig.html', width=1450, height=1000)

In [ ]:
ngram_proportion_fig = make_subplots(
    rows=num_rows, cols=num_cols, subplot_titles=unique_topics,
    vertical_spacing=0.1, horizontal_spacing=0.1
)

legend_added = {'unigram': False, 'bigram': False, 'trigram': False}
for i, custom_label in enumerate(unique_topics):
    topic_df = all_proportions_topic[all_proportions_topic['CustomName'] == custom_label]
    row = (i // num_cols) + 1
    col = (i % num_cols) + 1
    for ngram_type in ['unigram', 'bigram', 'trigram']:
        filtered_df = topic_df[topic_df['ngram_type'] == ngram_type]
        ngram_proportion_fig.add_trace(
            go.Scatter(
                x=filtered_df['Year'],
                y=filtered_df['proportion_new_ngrams'],
                mode='lines+markers',
                name=ngram_type.capitalize() if not legend_added[ngram_type] else None,
                line=dict(color=colors[ngram_type], width=0.9),
                marker=dict(size=3),
                hovertemplate=(
                    'N-gram Type: %{text}<br>'
                    'Year: %{x}<br>'
                    'Custom Label: %{customdata}<br>'
                    'Proportion: %{y}<br>'
                ),
                text=filtered_df['ngram_type'],
                customdata=[custom_label] * len(filtered_df),
                showlegend=not legend_added[ngram_type]
            ),
            row=row,
            col=col
        )
        legend_added[ngram_type] = True

ngram_proportion_fig.update_layout(
    width=num_cols * 400,
    height=num_rows * 250,
    showlegend=True,
    legend_title_text='N-gram Type',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.1,
        xanchor="center",
        x=0.5
    ),
    title_text="Unique N-gram Proportions Over Years by Topic",
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)

ngram_proportion_fig.add_annotation(
    x=-0.08, y=0.3,
    xref='paper', yref='paper',
    showarrow=False,
    text="New N-grams/Unique N-grams",
    textangle=-90
)

ngram_proportion_fig.update_xaxes(tickangle=-70, showline=True, linewidth=0.7, linecolor='black', showticklabels=True)
ngram_proportion_fig.update_yaxes(showline=True, linewidth=0.7, linecolor='black', showticklabels=True, title_text=None)

ngram_proportion_fig.write_html("results/ngram_proportion_fig.html")
ngram_proportion_fig.show()

In [106]:
IFrame(src='./results/ngram_proportion_fig.html', width=1450, height=1000)

In [107]:
df.to_pickle("files/df.pkl")

In [108]:
file_path = 'files/saved_data.pkl'

saved_data = {
    'df': df
}

with open(file_path, 'wb') as f:
    dill.dump(saved_data, f)

In [110]:
from nbconvert import HTMLExporter
import nbformat

notebook_path = 'index.ipynb'
html_exporter = HTMLExporter()

with open(notebook_path, 'r', encoding='utf-8') as nb_file:
    notebook_content = nb_file.read()
    notebook = nbformat.reads(notebook_content, as_version=4)

html_output, _ = html_exporter.from_notebook_node(notebook)

with open('index.html', 'w', encoding='utf-8') as html_file:
    html_file.write(html_output)